### Data Ingestion Pipeline: Parse documents, chunk them, and create embeddings to store in a vector database for retrieval.

In [6]:
### Document Data Structure: Page Content (str) + Metadata (dict)

from langchain_community.document_loaders import PyMuPDFLoader      # To load a PDF file
from langchain_community.document_loaders import DirectoryLoader    # To load multiple files at once

## Loads all the PDF files from the data directory and converts them into a list of Document objects, where each Document represents a single page from the PDF files, containing the page content and associated metadata.
dir_loader = DirectoryLoader(
    "/Users/atharva/Documents/GitHub/RAG_Pipeline/Data",                # Path to the directory containing the PDF files
    glob="**/*.pdf",                                                    # Load all PDF files in the directory and its subdirectories
    loader_cls=PyMuPDFLoader,                                           # Use PyMuPDFLoader to load PDF files
    show_progress= True                     
)
pdf_documents = dir_loader.load()                                       # Load the PDF documents into a list of Document objects
pdf_documents

print(f"Number of pages/ Documents loaded: {len(pdf_documents)}")       # Print the number of PDF documents loaded

# Notes
# 1. The pdf_documents variable is a list of Document objects, where each Document contains the content of a single page from the PDF files and its associated metadata.
# 2. Total Documents loaded = Number of pages in all the PDF files combined. Each page is treated as a separate Document object.
# Thus pdf_documents is a list of 215 Document objects

100%|██████████| 6/6 [00:02<00:00,  2.89it/s]

Number of pages/ Documents loaded: 215


In [7]:
### Chunking: From the above code, we have a list of Document objects, where each Document represents a single page from the PDF files. To make this data more manageable for retrieval and generation, we can apply chunking to break down the content into smaller pieces.
# So now we will be breaking down the information in the content section of the Document object into smaller chunks. This is important because it allows us to create more focused and relevant embeddings for each chunk, which can improve the retrieval performance when we search for relevant information based on user queries.


from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size: max number of characters per chunk
# chunk_overlap: number of characters shared between consecutive chunks so context is not lost at chunk boundaries
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Takes the list of page-level Documents and returns a new list of
# smaller chunk-level Documents, each inheriting the source metadata
chunks = text_splitter.split_documents(pdf_documents)

print(f"Total pages  : {len(pdf_documents)}")
print(f"Total chunks : {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0]}")

# chunks is again a list of Document objects, but now each Document represents a smaller chunk of text (up to 500 characters) from the original page content, along with the associated metadata. This allows us to create more focused embeddings for each chunk, which can improve retrieval performance when we search for relevant information based on user queries.

# Notes
# 1. The text_splitter got a list of 215 Document objects to further split into chunks of 500 characters each with an overlap of 50 characters
# 2. chunks variable is again a list of Documents objects, however, now each Document object includes 500 chars and associated metadata
# 3. The length of chunks = 1067, which means there are 1067 Document objects created

Total pages  : 215
Total chunks : 1067

Sample chunk:
page_content='CYBERCARDIA: PATIENT-SPECIFIC ELECTROPHYSIOLOGICAL HEART MODEL FOR
ASSISTING LEFT ATRIUM ARRHYTHMIA ABLATION
Jiyue He
A DISSERTATION
in
Electrical and Systems Engineering
Presented to the Faculties of the University of Pennsylvania
in
Partial Fulfillment of the Requirements for the
Degree of Doctor of Philosophy
2023
Supervisor of Dissertation
Rahul Mangharam, Professor of Electrical and Systems Engineering and Computer and Information
Science Departments
Graduate Group Chairperson' metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-02-01T01:58:55+00:00', 'source': '/Users/atharva/Documents/GitHub/RAG_Pipeline/Data/PFA_EP_Mapping_AFib_Comprehensive_Update.pdf', 'file_path': '/Users/atharva/Documents/GitHub/RAG_Pipeline/Data/PFA_EP_Mapping_AFib_Comprehensive_Update.pdf', 'total_pages': 137, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddat

### Creating Embedding and Storing in Vector Database

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer       # To create embeddings using a pre-trained model
import chromadb                                             # To create a vector database for storing embeddings
from chromadb.config import Settings
import uuid                                                 # To generate unique IDs for each chunk when storing in the vector database


from typing import List, Dict, Any                          # For type annotations to improve code readability and maintainability  
from sklearn.metrics.pairwise import cosine_similarity      # To calculate cosine similarity between query embedding and chunk embeddings for retrieval

In [ ]:
class EmbeddingManager:         # document embedding generation using Sentence Transformer
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):   # Initializes the EmbeddingManager class with a specified model name for generating embeddings. The default model is "all-MiniLM-L6-v2"
        # model_name: Hugging Face model name for generating embeddings; default is "all-MiniLM-L6-v2" which is a popular model for creating sentence embeddings.
        self.model_name = model_name    # Name of the pre-trained model to use for generating embeddings
        self.model = None               # Placeholder for the loaded model instance
        self._load_model()              # Load the model when the EmbeddingManager instance is created; this is a function call to the private method _load_model() which initializes the model attribute with the specified pre-trained model.
        
    def _load_model(self):              # Loads the Sentence Transformer model
        try:
            print(f"Loading model: {self.model_name}")                      # Print a message indicating that the model is being loaded
            self.model = SentenceTransformer(self.model_name)               # Load the specified pre-trained model using SentenceTransformer and assign it to the model attribute
            print(f"Model '{self.model_name}' loaded successfully with embedding dimension: {self.model.get_embedding_dimension()}")  
            
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")          # Print an error message if the model fails to load
            raise e                                                         # Raise the exception to indicate that the model loading failed
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:          # Generates embeddings for a list of texts using the loaded model
        
        if not self.model:
            raise ValueError("Model is not loaded. Please load the model before generating embeddings.")            # Raise an error if the model is not loaded

        print(f"Generating embeddings for {len(texts)} texts using model '{self.model_name}'")                      # Print a message indicating the number of texts for which embeddings are being generated
        
        embeddings = self.model.encode(texts, show_progress_bar=True)                                               # Generate embeddings for the input texts using the loaded model and show a progress bar during the process
        print(f"Embeddings generated successfully with shape: {embeddings.shape}")                                  # Print a message indicating that embeddings were generated successfully and display the shape of the resulting embeddings array
        return embeddings                                                                                           # Return the generated embeddings as a NumPy array
        
    
    
    
### Initialize the EmbeddingManager Class

embedding_manager = EmbeddingManager()              # Create an instance of the EmbeddingManager class
print(embedding_manager)

# Notes
# 1. The SentenceTransformer model create embeddings of size 384 for each chunk of text
# 2. For one chunk the numpy array of embeddings will look like this: [0.123, 0.456, 0.789, ..., 0.987] (size 384) - where each value is a float
# 3. In this case, our one chunk is a Document with 500 characters
# 4. Thus, 

Loading model: all-MiniLM-L6-v2
Model 'all-MiniLM-L6-v2' loaded successfully with embedding dimension: 384


### Vector Store

In [ ]:
class VectorStore:
    
    def __init__(self, collection_name: str = "EP_PDF_Documents", persist_directory: str = "./vector_store"):
        self.collection_name = collection_name          # Name of the collection in the vector store where embeddings will be stored - it is like a table which holds related embeddings together
        self.persist_directory = persist_directory      # Directory path where the vector store will persist its data; this allows for saving and loading the vector store across sessions
        self.client = None                              # client is the connection to the DB, it is an object used to create, access and manage collections in the DB
        self.collection = None
        self._initialize_vector_store()                 # Initialize the Chroma vector store client and collection; this is a function call to the private method _initialize_vector_store() which sets up the vector store for storing embeddings.
        
    def _initialize_vector_store(self):
        """ Initializes the Chroma vector store client and collection. If the collection does not exist, it creates a new one """
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok = True)                        # Checks if the directory exits on the given path
            self.client = chromadb.PersistentClient(path = self.persist_directory)      
            
            # Get or create a new collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"Description" : "PDF Document embeddings for RAG"}
            )
            print(f"Vector Store initialized. Collection : {self.collection_name}")
            print(f"Existing document objects in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error initializing the Vector Store: {e}")
            raise e
        
    def add_documents(self, documents:List[Any], embeddings: np.ndarray):
        """
        Add document objects to the Vector Store
        
        Args:
            documents: List of LangChain document objects
            embeddings: Corresponding embeddings of the documents
        """
        
        # No of Documents = 1067 (No of chunks created by text splitter)
        # The EmbeddingManager will create the embeddings of size (1067 x 384) - 1067 vectors of size 384
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to the Vector Store")
        
        # Prepare the data to store inside ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID: This is the ID for each document/ chunk
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_context)
            metadatas.append(metadata)
            
            
            
            
            
            